In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from scipy.spatial.distance import cosine, cityblock, euclidean, canberra, correlation, chebyshev
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split

import psutil
import gc

In [2]:
dtypes = {
    'offer_depersanalised': 'str',
    'goods_depersanalised': 'str',
    'goods_category_id': 'str',
    'id': 'str',
}

train = pd.read_csv('inter-task-binary-classification-offers-on-the-marketplace/train.csv', dtype=dtypes)
test = pd.read_csv('inter-task-binary-classification-offers-on-the-marketplace/test.csv', dtype=dtypes)

train.head()

,offer_depersanalised,goods_depersanalised,sum_length,attrs+title_score,offer_price,goods_price,goods_category_id,target,id
0,295140,1396793,37,0.027267,1070,NaN,14.0,0,295140$1396793
1,65291,1396586,38,0.050415,698,NaN,14.0,0,65291$1396586
2,39232,1396244,38,0.087280,837,NaN,14.0,0,39232$1396244
3,39232,1396513,38,0.087280,837,NaN,14.0,0,39232$1396513
4,65052,1396237,38,0.079773,1085,NaN,14.0,0,65052$1396237


In [3]:
print(train.shape)
print(test.shape)

(2518441, 9)
(363835, 8)


In [4]:
train.dtypes

offer_depersanalised     object
goods_depersanalised     object
sum_length                int64
attrs+title_score       float64
offer_price               int64
goods_price             float64
goods_category_id        object
target                    int64
id                       object
dtype: object

In [5]:
train.isna().sum()

offer_depersanalised         0
goods_depersanalised         0
sum_length                   0
attrs+title_score            0
offer_price                  0
goods_price             407287
goods_category_id          833
target                       0
id                           0
dtype: int64

In [6]:
test.isna().sum()

offer_depersanalised        0
goods_depersanalised        0
sum_length                  0
attrs+title_score           0
offer_price                 0
goods_price             58971
goods_category_id         131
id                          0
dtype: int64

In [7]:
train['is_train'] = 1
test['is_train']  = 0

all_data = pd.concat([train, test], ignore_index=True, sort=False)

In [8]:
offer_title_ids = np.load('inter-task-binary-classification-offers-on-the-marketplace/offer_title_vectors/offer_title_vectors/items_deperson.npy', allow_pickle=True)
offer_title_embeds = np.load('inter-task-binary-classification-offers-on-the-marketplace/offer_title_vectors/offer_title_vectors/embed_deperson.npy')
offer_title_dict = dict(zip(offer_title_ids, offer_title_embeds))

offer_image_ids = np.load('inter-task-binary-classification-offers-on-the-marketplace/offer_image_vectors/offer_image_vectors/items_deperson.npy', allow_pickle=True)
offer_image_embeds = np.load('inter-task-binary-classification-offers-on-the-marketplace/offer_image_vectors/offer_image_vectors/embed_deperson.npy')
offer_image_dict = dict(zip(offer_image_ids, offer_image_embeds))

goods_title_ids = np.load('inter-task-binary-classification-offers-on-the-marketplace/goods_title_vectors/goods_title_vectors/items_deperson.npy', allow_pickle=True)
goods_title_embeds = np.load('inter-task-binary-classification-offers-on-the-marketplace/goods_title_vectors/goods_title_vectors/embed_deperson.npy')
goods_title_dict = dict(zip(goods_title_ids, goods_title_embeds))

goods_image_ids = np.load('inter-task-binary-classification-offers-on-the-marketplace/goods_image_vectors/goods_image_vectors/items_deperson.npy', allow_pickle=True)
goods_image_embeds = np.load('inter-task-binary-classification-offers-on-the-marketplace/goods_image_vectors/goods_image_vectors/embed_deperson.npy')
goods_image_dict = dict(zip(goods_image_ids, goods_image_embeds))

In [9]:
all_data['offer_title'] = all_data['offer_depersanalised'].astype(str).map(offer_title_dict)
all_data['goods_title'] = all_data['goods_depersanalised'].astype(str).map(goods_title_dict)

all_data['offer_image'] = all_data['offer_depersanalised'].astype(str).map(offer_image_dict)
all_data['goods_image'] = all_data['goods_depersanalised'].astype(str).map(goods_image_dict)

In [10]:
del offer_title_ids, offer_title_embeds, offer_title_dict, offer_image_ids, offer_image_embeds, offer_image_dict, goods_title_ids, goods_title_embeds, goods_title_dict, goods_image_ids, goods_image_embeds, goods_image_dict

gc.collect()

88

In [11]:
n_title = 32
n_image = 64

# title
mask_offer_title = all_data['offer_title'].notna()
if mask_offer_title.sum() > 0:
    offer_title_valid = np.stack(all_data.loc[mask_offer_title, 'offer_title'])
    
    pca_offer_title = PCA(n_components=n_title)
    pca_offer_title.fit(offer_title_valid)
    
    offer_title_pca = np.full((len(all_data), n_title), np.nan)
    offer_title_pca[mask_offer_title] = pca_offer_title.transform(offer_title_valid)
    
    offer_title_pca_df = pd.DataFrame(offer_title_pca, columns=[f'pca_offer_title_{i}' for i in range(n_title)], index=all_data.index)

mask_goods_title = all_data['goods_title'].notna()
if mask_goods_title.sum() > 0:
    goods_title_valid = np.stack(all_data.loc[mask_goods_title, 'goods_title'])
    
    pca_goods_title = PCA(n_components=n_title)
    pca_goods_title.fit(goods_title_valid)
    
    goods_title_pca = np.full((len(all_data), n_title), np.nan)
    goods_title_pca[mask_goods_title] = pca_goods_title.transform(goods_title_valid)
    
    goods_title_pca_df = pd.DataFrame(goods_title_pca, columns=[f'pca_goods_title_{i}' for i in range(n_title)], index=all_data.index)

# image
mask_offer_image = all_data['offer_image'].notna()
if mask_offer_image.sum() > 0:
    offer_image_valid = np.stack(all_data.loc[mask_offer_image, 'offer_image'])
    
    pca_offer_image = PCA(n_components=n_image)
    pca_offer_image.fit(offer_image_valid)
    
    offer_image_pca = np.full((len(all_data), n_image), np.nan)
    offer_image_pca[mask_offer_image] = pca_offer_image.transform(offer_image_valid)
    
    offer_image_pca_df = pd.DataFrame(offer_image_pca, columns=[f'pca_offer_image_{i}' for i in range(n_image)], index=all_data.index)

mask_goods_image = all_data['goods_image'].notna()
if mask_goods_image.sum() > 0:
    goods_image_valid = np.stack(all_data.loc[mask_goods_image, 'goods_image'])
    
    pca_goods_image = PCA(n_components=n_image)
    pca_goods_image.fit(goods_image_valid)
    
    goods_image_pca = np.full((len(all_data), n_image), np.nan)
    goods_image_pca[mask_goods_image] = pca_goods_image.transform(goods_image_valid)
    
    goods_image_pca_df = pd.DataFrame(goods_image_pca, columns=[f'pca_goods_image_{i}' for i in range(n_image)], index=all_data.index)

all_data = all_data.join([offer_title_pca_df, goods_title_pca_df, offer_image_pca_df, goods_image_pca_df])

added_cols = (n_title * 2) + (n_image * 2)
print(f"added cols: {added_cols}")

added cols: 192


In [12]:
del offer_title_pca_df, goods_title_pca_df, offer_image_pca_df, goods_image_pca_df

gc.collect()

0

In [13]:
IMG_DIM = 256
TXT_DIM = 64

def safe_stack(series, dim):
    return np.stack([
        v if isinstance(v, np.ndarray) else np.zeros(dim)
        for v in series.values
    ])

offer_img = safe_stack(all_data['offer_image'], IMG_DIM)
goods_img = safe_stack(all_data['goods_image'], IMG_DIM)

offer_txt = safe_stack(all_data['offer_title'], TXT_DIM)
goods_txt = safe_stack(all_data['goods_title'], TXT_DIM)

def safe_corr(v1, v2):
    val = correlation(v1, v2)
    return 1.0 if np.isnan(val) else val

def safe_pearsonr(v1, v2):
    try:
        return pearsonr(v1, v2)[0]
    except:
        return 0.0

def safe_spearmanr(v1, v2):
    try:
        return spearmanr(v1, v2)[0]
    except:
        return 0.0


all_data['manhattan_title'] = [cityblock(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['euclidean_title'] = [euclidean(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['cosine_title'] = [cosine(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['correlation_title'] = [safe_corr(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['canberra_title'] = [canberra(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['pearson_title'] = [safe_pearsonr(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['spearman_title'] = [safe_spearmanr(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]
all_data['chebyshev_title'] = [chebyshev(v1, v2) for v1, v2 in zip(offer_txt, goods_txt)]

all_data['manhattan_image'] = [cityblock(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['euclidean_image'] = [euclidean(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['cosine_image'] = [cosine(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['correlation_image'] = [safe_corr(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['canberra_image'] = [canberra(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['pearson_image'] = [safe_pearsonr(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['spearman_image'] = [safe_spearmanr(v1, v2) for v1, v2 in zip(offer_img, goods_img)]
all_data['chebyshev_image'] = [chebyshev(v1, v2) for v1, v2 in zip(offer_img, goods_img)]

MemoryError: Unable to allocate 2.00 KiB for an array with shape (256,) and data type float64

In [ ]:
del offer_img, goods_img, offer_txt, goods_txt

gc.collect()

In [ ]:
all_data.columns

In [ ]:
# title * image
all_data['cosine_prod'] = all_data['cosine_title'] * all_data['cosine_image']
all_data['euclidean_prod'] = all_data['euclidean_title'] * all_data['euclidean_image']
all_data['manhattan_prod'] = all_data['manhattan_title'] * all_data['manhattan_image']
all_data['correlation_prod'] = all_data['correlation_title'] * all_data['correlation_image']
all_data['canberra_prod'] = all_data['canberra_title'] * all_data['canberra_image']
all_data['pearson_prod'] = all_data['pearson_title'] * all_data['pearson_image']
all_data['spearman_prod'] = all_data['spearman_title'] * all_data['spearman_image']
all_data['chebyshev_prod'] = all_data['chebyshev_title'] * all_data['chebyshev_image']


# attrs+title_score prod
all_data['score_cosine_title'] = all_data['attrs+title_score'] * all_data['cosine_title']
all_data['score_cosine_image'] = all_data['attrs+title_score'] * all_data['cosine_image']

all_data['score_euclidean_title'] = all_data['attrs+title_score'] * all_data['euclidean_title']
all_data['score_euclidean_image'] = all_data['attrs+title_score'] * all_data['euclidean_image']

all_data['score_manhattan_title'] = all_data['attrs+title_score'] * all_data['manhattan_title']
all_data['score_manhattan_image'] = all_data['attrs+title_score'] * all_data['manhattan_image']

all_data['score_correlation_title'] = all_data['attrs+title_score'] * all_data['correlation_title']
all_data['score_correlation_image'] = all_data['attrs+title_score'] * all_data['correlation_image']

all_data['score_canberra_title'] = all_data['attrs+title_score'] * all_data['canberra_title']
all_data['score_canberra_image'] = all_data['attrs+title_score'] * all_data['canberra_image']

all_data['score_pearson_title'] = all_data['attrs+title_score'] * all_data['pearson_title']
all_data['score_pearson_image'] = all_data['attrs+title_score'] * all_data['pearson_image']

all_data['score_spearman_title'] = all_data['attrs+title_score'] * all_data['spearman_title']
all_data['score_spearman_image'] = all_data['attrs+title_score'] * all_data['spearman_image']

all_data['score_chebyshev_title'] = all_data['attrs+title_score'] * all_data['chebyshev_title']
all_data['score_chebyshev_image'] = all_data['attrs+title_score'] * all_data['chebyshev_image']

In [ ]:
# price
all_data['price_diff'] = abs(all_data['offer_price'] - all_data['goods_price']).fillna(0)
all_data['price_ratio'] = all_data['offer_price'] / (all_data['goods_price'] + 1)
all_data['score_price_ratio'] = all_data['attrs+title_score'] * all_data['price_ratio']

all_data['log_offer_price'] = np.log1p(all_data['offer_price'])
all_data['log_goods_price'] = np.log1p(all_data['goods_price'])
all_data['log_price_diff'] = abs(all_data['log_offer_price'] - all_data['log_goods_price'])

all_data['score_log_price_diff'] = all_data['attrs+title_score'] * all_data['log_price_diff']
all_data['price_diff_log_ratio'] = all_data['log_price_diff'] / (all_data['log_offer_price'] + all_data['log_goods_price'] + 1)

# category mean target
category_mean = train.groupby('goods_category_id')['target'].mean()
all_data['cat_mean_target'] = all_data['goods_category_id'].map(category_mean).fillna(train['target'].mean())

# flags
all_data['has_both_images'] = ((all_data['offer_image'].notna()) & (all_data['goods_image'].notna())).astype(int)
all_data['is_much_cheaper'] = (all_data['price_ratio'] < 0.75).astype(int)
all_data['is_much_more_expensive'] = (all_data['price_ratio'] > 1.40).astype(int)

all_data['price_ratio_rank_in_offer'] = all_data.groupby('offer_depersanalised')['price_ratio'].rank(method='min', ascending=True)
all_data['price_ratio_rank_in_offer'] = all_data['price_ratio_rank_in_offer'].fillna(1.0)  # для одиночных/NaN — ранг 1

all_data['price_ratio_log'] = np.log1p(all_data['price_ratio'].clip(0.1, 10))

In [ ]:
all_data.isna().sum()

In [ ]:
all_data.drop(columns=['offer_depersanalised', 'goods_depersanalised', 'offer_title', 'goods_title', 'offer_image', 'goods_image'], inplace=True)

gc.collect()

In [ ]:
train = all_data[all_data['is_train'] == 1]
test = all_data[all_data['is_train'] == 0]

train = train.drop(columns=['is_train'])
test = test.drop(columns=['target', 'is_train'])

In [ ]:
del all_data

gc.collect()

In [ ]:
X = train.drop(columns=['target', 'id'])
y = train['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

model = CatBoostClassifier(
    iterations = 1000,
    eval_metric = 'F1',
    random_state = 42,
    early_stopping_rounds=100,
    verbose = 100,
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

# Оценка
val_pred = model.predict(X_val)
f1 = f1_score(y_val, val_pred)
print(f"F1 на валидации: {f1:.4f}")

In [ ]:
del X, y

gc.collect()

In [ ]:
test_proba = model.predict_proba(test[X_train.columns])[:, 1]

high_threshold = 0.99
low_threshold = 0.01

confident_pos_mask = test_proba > high_threshold
confident_neg_mask = test_proba < low_threshold

print(f"Уверенных positive: {confident_pos_mask.sum()} ({confident_pos_mask.mean():.2%})")
print(f"Уверенных negative: {confident_neg_mask.sum()} ({confident_neg_mask.mean():.2%})")

pseudo_data = test[confident_pos_mask | confident_neg_mask].copy()
pseudo_data['target'] = 0
pseudo_data.loc[confident_pos_mask[confident_pos_mask | confident_neg_mask], 'target'] = 1
pseudo_data['weight'] = 0.5

extended_train = pd.concat([train.copy(), pseudo_data], ignore_index=True)
extended_X = extended_train.drop(['target', 'id', 'weight'], axis=1, errors='ignore')
extended_y = extended_train['target']
extended_weights = extended_train['weight'].fillna(1.0)

model.fit(
    extended_X, extended_y,
    eval_set=(X_val, y_val),
    use_best_model=True
)

In [ ]:
del extended_weights, test_proba, confident_pos_mask, confident_neg_mask

gc.collect()

In [ ]:
model.save_model('models/final_model.cbm')

In [ ]:
val_proba_final = model.predict_proba(X_val)[:, 1]

def find_best_threshold(y_true, proba):
    prec, rec, thresh = precision_recall_curve(y_true, proba)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-12)
    
    best_idx = np.argmax(f1_scores)
    best_thresh = thresh[best_idx]
    
    return best_thresh

best_threshold = find_best_threshold(y_val, val_proba_final)

print("best threshold:", best_threshold)

In [ ]:
test_proba_final = model.predict_proba(test[model.feature_names_])[:, 1]
test_pred = (test_proba_final >= 0.41).astype(int)

submission = pd.DataFrame({
    'id': test['id'],
    'target': test_pred
})

submission.to_csv('results/final_submission.csv', index=False)

In [ ]:
importances = model.get_feature_importance(prettified=True)

print("Топ важных признаков:")
print(importances.head(50))

plt.figure(figsize=(12, 8))
plt.barh(importances['Feature Id'][:20][::-1], importances['Importances'][:20][::-1], color='skyblue')
plt.xlabel('Важность (feature importance)')
plt.title('Топ-20 самых важных признаков (CatBoost)')
plt.gca()
plt.tight_layout()
plt.show()